# BioNeMo Agentic Scientist: Autonomous Drug Discovery Tutorial

Welcome to the **BioNeMo Agentic Scientist** hands-on tutorial. In this notebook, we demonstrate how an autonomous multi-agent AI system designs, evaluates, and selects drug-like candidates against challenging oncology targets.

### Multi-Agent Pipeline Overview:
1. **Target Scout Agent**: Resolves biological targets (UniProt/PDB), identifies binding pockets, and verifies canonical residues.
2. **Generative Chemist Agent**: Synthesizes bioisosteric derivatives exploring latent chemical space (powered by NVIDIA NIM MolMIM).
3. **ADMET Critic Agent**: Evaluates Lipinski Rule of 5, Veber oral bioavailability, QED, Synthetic Accessibility (SAscore), and PAINS toxicophore substructures via RDKit.
4. **Biophysics Docking Agent**: Predicts 3D binding poses and affinities (Delta G) against target crystal structures (powered by NVIDIA NIM DiffDock).
5. **Principal Investigator (PI) Agent**: Orchestrates multi-objective Pareto frontier ranking and coordinates an autonomous feedback loop.

In [ ]:
# Setup sys.path to load local source modules
import sys, os
sys.path.insert(0, os.path.abspath('..'))

from src.target_scout import TargetScoutAgent
from src.generative_chemist import GenerativeChemistAgent
from src.admet_critic import ADMETCriticAgent
from src.docking_agent import BiophysicsDockingAgent
from src.pi_agent import PrincipalInvestigatorAgent
from src.orchestrator import AgenticScientistOrchestrator
from src.visualizer import plot_pareto_frontier, plot_chemical_leads_grid

print("BioNeMo Agentic Scientist modules loaded successfully!")

## 1. Target Scout: Resolving Target & Binding Pocket

In [ ]:
scout = TargetScoutAgent()
target_profile, scout_msg = scout.scout_target('KRAS G12D')

print(f"Target Resolved: {target_profile.name} ({target_profile.gene})")
print(f"PDB Crystal Structure: {target_profile.pdb_id}")
print(f"Reference Ligand: {target_profile.reference_ligand_name}")
print(f"Key Pocket Residues: {', '.join(target_profile.pocket_residues)}")
print(f"Agent Thought:\n{scout_msg.thought}")

## 2. Generative Chemist: Latent Space Exploration (MolMIM)
Generating bioisosteric derivatives around the parent scaffold.

In [ ]:
chemist = GenerativeChemistAgent(mock=True)
candidates, chem_msg = chemist.generate_derivatives(
    parent_smiles=target_profile.reference_ligand_smiles,
    target_name=target_profile.name,
    num_molecules=8,
    round_num=1
)

print(f"Generated {len(candidates)} candidates.")
print(f"Candidate 1 ID: {candidates[0].id}")
print(f"Candidate 1 SMILES: {candidates[0].smiles}")

## 3. ADMET Critic: Cheminformatics & Physicochemical Profiling

In [ ]:
critic = ADMETCriticAgent()
screened_cands, critic_msg = critic.evaluate_candidates(candidates)

print("ID | MW | LogP | QED | SAscore | PAINS | Verdict")
print("-" * 55)
for c in screened_cands:
    pains_str = 'None' if not c.pains_alerts else ','.join(c.pains_alerts)
    print(f"{c.id} | {c.mw:.1f} | {c.logp:.2f} | {c.qed:.2f} | {c.sascore:.2f} | {pains_str} | {c.admet_verdict}")

## 4. Biophysics Docking: 3D Binding Conformation & Affinity (DiffDock)

In [ ]:
docking = BiophysicsDockingAgent(mock=True)
docked_cands, dock_msg = docking.dock_candidates(screened_cands, target_profile)

print(f"Successfully docked {len(docked_cands)} molecules into {target_profile.name} pocket.")
for c in docked_cands[:3]:
    print(f"{c.id}: Affinity = {c.binding_affinity:.2f} kcal/mol, DiffDock Confidence = {c.diffdock_confidence:.2f}")

## 5. Principal Investigator (PI) Agent: Multi-Parameter Pareto Optimization

In [ ]:
pi = PrincipalInvestigatorAgent()
top_leads, pi_msg = pi.evaluate_and_rank_leads(docked_cands, target_profile)

print(f"Nominated {len(top_leads)} top clinical leads.")
for rank, lead in enumerate(top_leads, 1):
    pareto = 'Pareto Optimal' if lead.is_pareto_optimal else 'Non-frontier'
    print(f"Rank #{rank}: {lead.id} | Affinity: {lead.binding_affinity:.2f} kcal/mol | QED: {lead.qed:.2f} | Score: {lead.composite_rank_score:.3f} | [{pareto}]")

## 6. End-to-End Orchestrated Campaign Execution

In [ ]:
orchestrator = AgenticScientistOrchestrator(mock=True)
dossier = orchestrator.run_discovery_campaign(
    target_query='EGFR T790M',
    num_candidates=6,
    enable_feedback_loop=True,
    output_dir='../results'
)

print("Campaign Completed!")
print(f"Screened: {dossier.screened_count} molecules")
print(f"Passed ADMET: {dossier.passed_admet_count} molecules")
print(f"Pareto Leads: {dossier.pareto_leads_count} molecules")
print("\nExecutive Summary:")
print(dossier.executive_summary)